In [1]:
# For automatic reloading of project modules
%load_ext autoreload
%autoreload 2

# Allow imports from the parent project folder
import sys
sys.path.append('..')

import numpy as np
import pandas as pd

import stoch_models
import numerical_schemes
import option_contracts
import analytical_results
import greeks

In [2]:
# Model parameters
model = stoch_models.GBM(r=0.05, sigma=0.2)

# Market and contract parameters
S = 100.0
t = 0.0
T = 1.0
K = 100.0
tau = T - t

option = option_contracts.EuropeanCall(
    strike=K,
    maturity=T,
)

# Simulation parameters
N = 100_000
M = 1
dx = 0.001
scheme = numerical_schemes.exact_step

times = np.linspace(t, T, M + 1)

# Common random numbers for all perturbed valuations
rng = np.random.default_rng(42)
Z = rng.standard_normal(size=(N, M))

In [3]:
result_delta = greeks.central_diff_delta(
    model=model,
    option_contract=option,
    scheme=scheme,
    S=S,
    N=N,
    times=times,
    dx=dx,
    normal_draws=Z,
)

result_vega = greeks.central_diff_vega(
    model=model,
    option_contract=option,
    scheme=scheme,
    S=S,
    N=N,
    times=times,
    dx=dx,
    normal_draws=Z,
)

result_gamma = greeks.central_diff_gamma(
    model=model,
    option_contract=option,
    scheme=scheme,
    S=S,
    N=N,
    times=times,
    dx=dx,
    normal_draws=Z,
)

In [4]:
bs_delta = analytical_results.black_scholes_delta(
    model=model, option_contract=option, S=S, t=t
)

bs_vega = analytical_results.black_scholes_vega(
    model=model, option_contract=option, S=S, t=t
)

bs_gamma = analytical_results.black_scholes_gamma(
    model=model, option_contract=option, S=S, t=t
)

In [5]:
results = [
    (result_delta, bs_delta),
    (result_vega, bs_vega),
    (result_gamma, bs_gamma),
]

# Table header
header = (
    f"{'Greek':<8}"
    f"{'Analytical':>14}"
    f"{'Monte Carlo':>14}"
    f"{'Error':>12}"
    f"{'Std. Error':>14}"
    f"{'CI Lower':>14}"
    f"{'CI Upper':>14}"
    f"{'Bump Size':>12}"
)

print(header)
print("-" * len(header))

# Table rows
for result, benchmark in results:
    print(
        f"{result.greek_name.capitalize():<8}"
        f"{benchmark:>14.6f}"
        f"{result.estimate:>14.6f}"
        f"{result.estimate - benchmark:>12.6f}"
        f"{result.standard_error:>14.6f}"
        f"{result.ci_lower:>14.6f}"
        f"{result.ci_upper:>14.6f}"
        f"{result.bump_size:>12.6f}"
    )

Greek       Analytical   Monte Carlo       Error    Std. Error      CI Lower      CI Upper   Bump Size
------------------------------------------------------------------------------------------------------
Delta         0.636831      0.634534   -0.002297      0.001823      0.630961      0.638107    0.100000
Vega         37.524035     37.480075   -0.043960      0.241220     37.007283     37.952867    0.000200
Gamma         0.018762      0.018182   -0.000580      0.001075      0.016075      0.020289    0.100000
